# Task 3 — Sentiment & Correlation Analysis

**Objective:** Combine sentiment scores from financial news headlines with stock returns and technical indicators, then analyse correlations using statistical and visual methods.

**Learning goals:**
- Apply VADER and TextBlob sentiment analysis to real text data
- Merge sentiment data with market data on a timeline
- Compute and interpret Pearson/Spearman correlations
- Produce a professional correlation report

## 1. Setup

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

from src.data_loader import fetch_stock_prices, load_csv
from src.sentiment import score_headlines
from src.indicators import daily_returns, rsi
from src.visualization import sentiment_vs_returns, correlation_heatmap

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 2. Load & Score Headlines

We attempt to load a headlines dataset. If none is available, we create a small synthetic example so the notebook remains fully runnable.

In [ ]:
headlines_path = root / "data" / "raw" / "headlines.csv"

if headlines_path.exists():
    headlines = load_csv("headlines.csv")
else:
    print("No headlines.csv found. Generating synthetic headlines for demonstration.")
    np.random.seed(42)
    dates = pd.date_range("2023-06-01", "2024-12-31", freq="D")
    n = len(dates)
    positive = [
        "Strong earnings beat estimates",
        "Company announces record revenue",
        "New product launch exceeds expectations",
        "Analyst upgrades stock to buy",
        "Market share continues to grow",
    ]
    negative = [
        "Profit warning sends shares lower",
        "Regulatory probe announced",
        "Earnings miss analyst forecasts",
        "Competitor gains market ground",
        "Supply chain disruptions persist",
    ]
    neutral = [
        "Company holds annual shareholder meeting",
        "New office opened in Chicago",
        "Board appoints new committee members",
        "Quarterly dividend declared",
        "Partnership agreement signed",
    ]

    all_headlines = positive + negative + neutral
    chosen = np.random.choice(all_headlines, size=n)
    headlines = pd.DataFrame({"date": dates, "headline": chosen, "ticker": "AAPL"})
    print(f"Created synthetic dataset with {len(headlines)} rows.")

headlines.head()

## 3. Compute Sentiment Scores

In [ ]:
headlines = score_headlines(headlines, text_column="headline")
headlines[["headline", "sentiment_vader", "sentiment_textblob"]].head(10)

## 4. Aggregate Sentiment by Day

We average sentiment scores per date so we can merge with daily stock returns.

In [ ]:
if "date" in headlines.columns:
    headlines["date"] = pd.to_datetime(headlines["date"])
    daily_sentiment = (
        headlines.groupby("date")[["sentiment_vader", "sentiment_textblob"]]
        .mean()
        .rename(columns={"sentiment_vader": "vader_avg", "sentiment_textblob": "textblob_avg"})
    )
else:
    daily_sentiment = headlines[["sentiment_vader", "sentiment_textblob"]].copy()
    daily_sentiment.columns = ["vader_avg", "textblob_avg"]

print(f"Daily sentiment shape: {daily_sentiment.shape}")
daily_sentiment.head()

## 5. Load Stock Prices & Compute Returns

In [ ]:
prices_path = root / "data" / "raw" / "stock_prices.csv"

if prices_path.exists():
    prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)
else:
    prices = fetch_stock_prices(["AAPL", "GOOG", "META"], start="2023-01-01")

# Focus on AAPL for correlation analysis
ticker = "AAPL" if "AAPL" in prices.columns else prices.columns[0]
close = prices[ticker].dropna()
ret = daily_returns(close).rename(f"{ticker}_return")

# Add RSI as a technical feature
rsi_vals = rsi(close, window=14).rename(f"{ticker}_rsi")

market_data = pd.DataFrame({f"{ticker}_close": close, f"{ticker}_return": ret, f"{ticker}_rsi": rsi_vals})
market_data.head()

## 6. Merge Sentiment & Market Data

In [ ]:
combined = market_data.join(daily_sentiment, how="inner")
combined = combined.dropna(subset=[f"{ticker}_return", "vader_avg"])
print(f"Combined dataset: {combined.shape[0]} rows, {combined.shape[1]} columns")
combined.head()

## 7. Visualise Sentiment vs. Returns

In [ ]:
fig = sentiment_vs_returns(
    combined,
    sentiment_col="vader_avg",
    return_col=f"{ticker}_return",
    title=f"{ticker} — VADER Sentiment vs. Daily Returns",
)
plt.show()

## 8. Correlation Analysis

In [ ]:
fig = correlation_heatmap(combined, title=f"{ticker} — Correlation Matrix (Sentiment + Market)")
plt.show()

In [ ]:
# Compute explicit Pearson and Spearman correlations
sentiment_cols = ["vader_avg", "textblob_avg"]
target_cols = [f"{ticker}_return", f"{ticker}_rsi"]

rows = []
for s in sentiment_cols:
    for t in target_cols:
        clean = combined[[s, t]].dropna()
        p_rho, p_p = pearsonr(clean[s], clean[t])
        s_rho, s_p = spearmanr(clean[s], clean[t])
        rows.append({
            "sentiment": s,
            "target": t,
            "pearson_r": round(p_rho, 4),
            "pearson_pval": round(p_p, 6),
            "spearman_rho": round(s_rho, 4),
            "spearman_pval": round(s_p, 6),
        })

corr_report = pd.DataFrame(rows)
corr_report

**Interpretation guide:**
- A positive Pearson / Spearman coefficient means that as sentiment increases, returns also tend to increase.
- The p-value tests whether the correlation is statistically significant (p < 0.05 is the conventional threshold).
- Lagged correlations (sentiment today vs. tomorrow's return) can be explored by shifting the sentiment column.

## 9. Lagged Correlation (Sentiment Leading Returns by 1 Day)

In [ ]:
lagged = combined.copy()
lagged["vader_lag1"] = lagged["vader_avg"].shift(1)
lagged = lagged.dropna(subset=["vader_lag1", f"{ticker}_return"])

p_rho, p_p = pearsonr(lagged["vader_lag1"], lagged[f"{ticker}_return"])
s_rho, s_p = spearmanr(lagged["vader_lag1"], lagged[f"{ticker}_return"])

print(f"=== Lagged Correlation (Sentiment[t-1] vs Return[t]) ===")
print(f"Pearson   r = {p_rho:.4f}  (p = {p_p:.6f})")
print(f"Spearman ρ = {s_rho:.4f}  (p = {s_p:.6f})")

## 10. Summary

- VADER and TextBlob provide quick, lexicon-based sentiment scores for financial headlines.
- Merging daily sentiment averages with stock returns enables correlation analysis.
- The heatmap and scatter plots reveal any linear/non-linear relationships.
- Lagged analysis tests whether news sentiment predicts next-day returns.
- This pipeline can be extended with more sophisticated NLP models (e.g., FinBERT) for better accuracy.